# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karim-yasser/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!git clone https://github.com/karim-yasser/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 174, done.
remote: Counting objects: 100% (174/174), done.
remote: Compressing objects: 100% (131/131), done.
remote: Total 174 (delta 78), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (174/174), 1.87 MiB | 6.64 MiB/s, done.
Resolving deltas: 100% (78/78), done.


In [3]:
import pandas as pd

path = "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(path)

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [4]:
X = df[
    [
        "content_age_days",
        "ctr",
        "search_volume",
        "competition",
        "word_count",
        "cpc"
    ]
]

y = ((df["content_age_days"] > 365) | (df["ctr"] < 2)).astype(int)

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Paper Finding 1

The paper reports that refreshing outdated content is associated with better search performance.

**My methodology question:**
How was "improvement" measured? Was the comparison made using the same time window and the same validation method for all pages?

---

## Paper Finding 2

The paper shows that multiple search signals can be used together to prioritize content updates.

**My methodology question:**
How were the labels created? Were they based on historical observations only, and were future signals completely excluded to avoid leakage?

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [5]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

In [13]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Train the model
model = DecisionTreeClassifier(
    random_state=0,
    max_depth=4
)

model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Accuracy
grouped_accuracy = accuracy_score(y_test, y_pred)*100

print("Grouped Accuracy:", grouped_accuracy ,"%")

Grouped Accuracy: 100.0 %


In [16]:
import pandas as pd

comparison = pd.DataFrame({
    "Validation": ["Random Split", "Grouped by Client"],
    "Accuracy(%)": [100.0, grouped_accuracy]
})

comparison

,Validation,Accuracy(%)
0,Random Split,100.0
1,Grouped by Client,100.0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## Leakage Audit

The model does not directly use the target label as an input feature.

However, the target labels were created using rule-based conditions that depend on features such as `content_age_days` and `ctr`.

Because these same features are used for training, the Decision Tree can learn the underlying rules very effectively. This likely explains the high accuracy observed during validation.

The grouped-by-client split reduces client overlap, but additional validation on future real-world labels would provide stronger evidence of generalization.

In [17]:
print("Features used in the model:")
print(X.columns.tolist())

Features used in the model:
['content_age_days', 'ctr', 'search_volume', 'competition', 'word_count', 'cpc']


### Leakage Review

The current feature set does not intentionally include the target label.

No future information or post-decision columns were used.

Even though the model achieved perfect accuracy, this may be because the target follows simple rule-based patterns that are easy for the model to learn.

Additional validation on new clients or future data would provide stronger evidence.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*


### Original Claim

The Decision Tree model achieved 100% accuracy and is better than the Week 4 baseline.

## Revised Claim

The Decision Tree achieved the same measured accuracy as the Week 4 baseline under the evaluated validation settings.

This result suggests that the model successfully learned the rule-based patterns present in the available data.

Because the target labels were generated from rules based on input features, these results should be interpreted as decision-support evidence rather than proof that the model will generalize to all future datasets.

This result suggests that the model successfully learned the observed rule-based patterns in the available dataset.

Because the target labels are generated from rule-based conditions and the dataset may contain highly correlated features, these results should be interpreted as decision-support evidence rather than proof that the model will generalize to future unseen data.

In [18]:
print(comparison)

          Validation  Accuracy(%)
0       Random Split        100.0
1  Grouped by Client        100.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [19]:
print(df.loc[train_idx, "client_id"].nunique())
print(df.loc[test_idx, "client_id"].nunique())

25
7


In [20]:
train_clients = set(df.loc[train_idx, "client_id"])
test_clients = set(df.loc[test_idx, "client_id"])

print(len(train_clients & test_clients))

0


In [22]:
print(X.columns.tolist())

['content_age_days', 'ctr', 'search_volume', 'competition', 'word_count', 'cpc']
